# L4b Algorithm: Depth-First Search

Depth-first search (DFS) explores one directed branch as far as it can before returning to the most recent vertex with an unfinished neighbor. In a recursive implementation, the active function calls themselves form the last-in, first-out stack that remembers where the search must return.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
> * __Trace recursive graph traversal:__ Follow the active call stack, visited set, and first-visit order as depth-first search enters a vertex and explores its outgoing neighbors. Identify where the traversal backtracks to an unfinished branch.
> * __Explain termination on a graph:__ Show why the visited-set guard turns cycles and converging edges into skipped recursive calls. Use that guard to explain why each reachable vertex is processed once.
> * __Connect the invariant to an implementation:__ Map state initialization, recursive first visits, and the initial call to the first three tasks in the student traversal file. Relate those stages to the running time and recursion storage.

Let's get started!

___

## The Algorithm

The figure shows the directed graph used in the L4b lab. Its sorted adjacency list begins with `1 => [2, 3]`, so the recursive search explores the branch beginning at vertex 2 before considering vertex 3 directly from vertex 1.

<div>
    <center>
        <img src="figs/Fig-Example-Graph.svg" width="480" alt="Six-vertex directed graph used for the depth-first-search trace"/>
    </center>
</div>

> __DFS recursion invariant__
>
> Every active call corresponds to one vertex on the current search branch. A first visit marks that vertex and records it before any outgoing neighbor is explored. When a call has no unseen neighbor left, it returns to its caller; that backtracking exposes the next unfinished neighbor on the branch. A call for an already visited vertex returns immediately.

__Initialization:__ Given an adjacency list for a directed graph and a starting vertex $v_s$, create an empty visited set and an empty first-visit order.

Define a recursive helper called [the `visit(...)` function](src/Compute.jl):

1. If $v$ is already visited, return without changing the traversal state.
2. Mark $v$ visited and append it to the first-visit order.
3. Examine the outgoing neighbors of $v$ in ascending identifier order, recursively calling [the `visit(...)` function](src/Compute.jl) for each one.
4. Return after every outgoing neighbor has been considered.

Call the recursive helper once with starting vertex $v_s$. The recursion ends when every branch from the starting vertex reaches either a vertex with no outgoing edge or a vertex already in the visited set.

___

## Trace the Recursion

Starting at vertex 1 gives the following state transitions. The stack is written from the oldest active call on the left to the newest call on the right. The first-visit order changes only when DFS enters an unseen vertex; backtracking and skipped calls leave it unchanged.

| Step | Action | Active call stack after the action | Visited vertices | First-visit order |
|:--|:--|:--|:--|:--|
| 1 | Visit `1` | `[1]` | `{1}` | `[1]` |
| 2 | Visit first neighbor `2` | `[1, 2]` | `{1, 2}` | `[1, 2]` |
| 3 | Visit first neighbor `3` | `[1, 2, 3]` | `{1, 2, 3}` | `[1, 2, 3]` |
| 4 | Visit neighbor `5` | `[1, 2, 3, 5]` | `{1, 2, 3, 5}` | `[1, 2, 3, 5]` |
| 5 | Visit neighbor `4` | `[1, 2, 3, 5, 4]` | `{1, 2, 3, 4, 5}` | `[1, 2, 3, 5, 4]` |
| 6 | Visit neighbor `6` | `[1, 2, 3, 5, 4, 6]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 5, 4, 6]` |
| 7 | Return from `6`, `4`, `5`, and `3` | `[1, 2]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 5, 4, 6]` |
| 8 | Skip already visited neighbor `4`; return from `2` | `[1]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 5, 4, 6]` |
| 9 | Skip already visited neighbor `3`; return from `1` | `[]` | `{1, 2, 3, 4, 5, 6}` | `[1, 2, 3, 5, 4, 6]` |

The edge `5 → 4` reaches vertex 4 before DFS returns to vertex 2 and considers the separate edge `2 → 4`. The visited-set guard therefore turns the later recursive request for vertex 4 into an immediate return. The same guard would stop a back edge in a directed cycle.

The resulting sequence `[1, 2, 3, 5, 4, 6]` happens to follow graph edges at each consecutive position for this example, but that is not guaranteed by DFS in general: after backtracking, the next first visit can begin from an earlier vertex on the active branch.

___

## Implementation Contract and Cost

The first three TODOs in [`src/Compute.jl`](src/Compute.jl) divide the trace into implementation stages: allocate the shared state, define the recursive first-visit operation, and start the recursion at the validated vertex. The nested operation can update `visited` and `order` because both collections belong to the surrounding function scope.

Let $V_r$ and $E_r$ denote the vertices and directed edges reachable from the start. Once outgoing-neighbor lists are ordered, depth-first search visits each reachable vertex once and examines each reachable edge once, giving $\mathcal{O}(|V_r|+|E_r|)$ traversal work. The lab's general interface also sorts a copy of each neighbor collection to guarantee deterministic output; that normalization contributes $\sum_{v\in V_r}\mathcal{O}(d_v\log d_v)$ work, where $d_v$ is the out-degree of vertex $v$.

The visited set and result require $\mathcal{O}(|V_r|)$ storage. The active recursion stack can also contain as many as $|V_r|$ calls when the reachable graph is one long directed chain. The algorithm terminates because a vertex is recorded only on its first visit and recursive calls eventually return from every finite neighbor list.

___

## Summary

Depth-first search uses recursive calls to remember an unfinished branch, recording each reachable vertex on first entry and backtracking when that branch cannot continue.

> __Key Takeaways:__
>
> * __The call stack is the DFS worklist:__ Each active call records one vertex on the current branch. Returning from a completed call restores the most recent vertex that may still have an unexplored neighbor.
> * __The visited guard guarantees termination:__ A recursive request for an already visited vertex returns immediately. That rule handles converging edges and prevents a directed cycle from producing infinite recursion.
> * __First-visit order depends on deterministic choices:__ The starting vertex, edge directions, and neighbor order determine which branch is explored first. Sorting outgoing neighbors makes that order reproducible, but it is still a traversal order rather than a guaranteed path.

Return to [the L4b lab](CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) and use this recursion invariant to complete TODOs 1 through 3 in [`src/Compute.jl`](src/Compute.jl).

___